### Use Obisidian Web Clipper browser to convert html file to markdown

### *I haven't gotten this anywhere near working*

From the browser, the Clipper semms to work more reliably than anything else I've tried. 

Snce NotebookLM accepts markdown, and since no matter what you give it, the first thing it does is to strip out anything but text, this seems to be a good solution.

Code started from [perplexity](https://www.perplexity.ai/search/is-there-a-way-to-test-a-chrom-oe__SyvZT_G2rMP6NvDBlw#3).

The environment needed to run this is tested in 

#### Installing and building obsidian web clipper

1. Download zip file from [here](https://github.com/obsidianmd/obsidian-clipper?tab=readme-ov-file)

2. unzip it whereever you want the compiled thing to be
3. cd to the unzippped dir: obsidian-clipper-main
4. install webpack: `npm install --save-dev webpack webpack-cli`
5. run:  `npm run build`
5. I also had to install node. chromedriver and something else, I think

In [1]:
import pathlib as pl
import sys
import os
from icecream import ic

extension_path = pl.Path(r'c:/Users/scott/OneDrive/share/ref/refwrangle/bin/chrome_extensions'
                         r'/obsidian-clipper-main/dist')

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser()
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw # assumed to be in same dir as this

html_path = pl.Path(r'C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/lit/lit_sources/')

html_file_path = html_path / 'Tumulty24FrischLearnedDemsShould.html'  # works, but complex WA post page has tons of extraneous link junk
#html_file_path = hpath / 'Walther24barstoolConservatism.html'     # works
#html_file_path = hpath / 'Yan24berkeleyFuncCallLeaderBrd.html'     # works now, used to fail on all b/c .html was corrupt

output_md_path = refwrangle_dir / "test/tmp_obswebclip.md"

# Get absolute path to HTML file
#html_path = os.path.abspath("test.html")
#file_url = f"file:///{html_path}"

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import os
import time

def clip_html_to_markdown(html_input_file, markdown_output_file, extension_path):
    """
    Clips content from an HTML file and saves it as a markdown file.
    
    Parameters:
        html_input_file (str): Path to the input HTML file
        markdown_output_file (str): Path to the output markdown file
        extension_path (str): Path to the built extension directory
    """
    chrome_options = Options()
    chrome_options.add_argument(f"--load-extension={extension_path}")
    driver = webdriver.Chrome(options=chrome_options)

    try:
        # Load the HTML file
        html_path = os.path.abspath(html_input_file)
        driver.get(f"file:///{html_path}")
        
        # Wait for extension to fully load
        time.sleep(2)
        
        # Find all extension-related elements
        extension_elements = driver.find_elements("xpath", 
            "//*[contains(@class, 'obsidian') or contains(@id, 'obsidian')]")
        
        if not extension_elements:
            raise Exception("No Obsidian extension elements found")
            
        # Find the clipper button (first clickable element)
        clipper = None
        for element in extension_elements:
            if element.is_displayed() and element.is_enabled():
                clipper = element
                break
                
        if not clipper:
            raise Exception("Could not find clickable clipper element")
            
        # Click the clipper
        clipper.click()
        
        # Wait for markdown preview
        markdown_preview = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "markdown-preview"))
        )
        
        # Get the markdown content
        markdown_content = markdown_preview.get_attribute("innerText")
        
        # Save to file
        with open(markdown_output_file, "w", encoding="utf-8") as f:
            f.write(markdown_content)
            
    finally:
        driver.quit()


In [3]:
clip_html_to_markdown(
            html_file_path,
            output_md_path,
            extension_path
        )

Exception: No Obsidian extension elements found

In [2]:
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.common.by import By
# import os

# def debug_clip_html_to_markdown(html_input_file, extension_path):
#     chrome_options = Options()
#     chrome_options.add_argument(f"--load-extension={extension_path}")
#     driver = webdriver.Chrome(options=chrome_options)

#     try:
#         html_path = os.path.abspath(html_input_file)
#         driver.get(f"file:///{html_path}")
        
#         # Wait for the page to load
#         import time
#         time.sleep(5)
        
#         # Get and print browser console logs
#         logs = driver.get_log('browser')
#         print("Browser Console Logs:")
#         for log in logs:
#             print(log)
            
#         # List all extension-related elements
#         print("\nExtension Elements:")
#         extension_elements = driver.find_elements("xpath", "//*[contains(@class, 'obsidian') or contains(@id, 'obsidian')]")
#         for element in extension_elements:
#             print(f"Tag: {element.tag_name}")
#             print(f"ID: {element.get_attribute('id')}")
#             print(f"Class: {element.get_attribute('class')}")
            
#         # Check if extension is loaded in Chrome
#         print("\nLoaded Extensions:")
#         print(driver.execute_script("return window.navigator.userAgent"))
        
#         input("Press Enter to close the browser...")
        
#     finally:
#         driver.quit()


In [7]:
# import unittest
# import tempfile
# import os
# import shutil

# class TestObsidianClipper(unittest.TestCase):
#     def setUp(self):
#         self.test_dir = tempfile.mkdtemp()
#         #self.extension_path = os.path.abspath("/path/to/built/extension/dist")
#         self.extension_path = extension_path # os.path.abspath("/path/to/built/extension/dist")
        
#         # Create test HTML file
#         self.test_html = os.path.join(self.test_dir, "test.html")
#         with open(self.test_html, "w", encoding="utf-8") as f:
#             f.write("""
#             <html>
#                 <body>
#                     <h1>Test Content</h1>
#                     <p>This is a test paragraph.</p>
#                 </body>
#             </html>
#             """)
        
#         self.output_markdown = os.path.join(self.test_dir, "output.md")

#     def test_clip_html_to_markdown(self):
#         clip_html_to_markdown(
#             self.test_html,
#             self.output_markdown,
#             self.extension_path
#         )
        
#         # Verify output file exists
#         self.assertTrue(os.path.exists(self.output_markdown))
        
#         # Check content
#         with open(self.output_markdown, "r", encoding="utf-8") as f:
#             content = f.read()
#             self.assertIn("# Test Content", content)
#             self.assertIn("This is a test paragraph", content)

#     def tearDown(self):
#         shutil.rmtree(self.test_dir)

# if __name__ == "__main__":
#     unittest.main()
